# Types of Crossmatches

In this tutorial, we will:

- explain what `how='inner'` (the default) and `how='left'` mean when crossmatching sky catalogs
- show which catalog is the "left" catalog and which is the "right" catalog
- filter the right catalog to create a realistic scenario where some sources have no counterpart
- demonstrate how each join type affects the row count and column values in the result
- give guidance on when to choose each option

In [1]:
import warnings

import lsdb

warnings.filterwarnings("ignore")

## 1. Open two catalogs with a small cone search

We open ZTF DR22 (left catalog) and Gaia DR3 (right catalog) restricted to a 36-arcsecond
cone so the example stays fast.

In [2]:
ztf22_sm = lsdb.open_catalog(
    "https://data.lsdb.io/hats/ztf_dr22",
    columns=["objectid", "objra", "objdec", "nepochs"],
    search_filter=lsdb.ConeSearch(280, 0, radius_arcsec=36),
)

gaia3_sm = lsdb.open_catalog(
    "s3://stpubdata/gaia/gaia_dr3/public/hats",
    columns=["source_id", "ra", "dec", "phot_g_mean_mag"],
    search_filter=lsdb.ConeSearch(280, 0, radius_arcsec=36),
)

In [3]:
n_ztf = len(ztf22_sm.compute())
n_gaia = len(gaia3_sm.compute())
print(f"ZTF sources in cone  : {n_ztf}")
print(f"Gaia sources in cone : {n_gaia}")

ZTF sources in cone  : 129
Gaia sources in cone : 17


## 2. Which catalog is "left" and which is "right"?

When you write

```python
ztf22_sm.crossmatch(gaia3_sm, ...)
```

- **`ztf22_sm` is always the left catalog** — the catalog you call `.crossmatch()` on.
- **`gaia3_sm` is always the right catalog** — the catalog you pass as the first argument.

The crossmatch algorithm iterates over every source in the *left* catalog and searches
for its nearest neighbor in the *right* catalog.  The `how` parameter controls what
happens to left-catalog sources that have no counterpart in the right catalog within
the search radius.

## 3. Filter the right catalog

Suppose we only want to match ZTF sources against *bright* Gaia stars
(magnitude < 20), for example to calibrate photometry against well-measured reference stars.
We filter Gaia with `.query()`, which leaves only 6 of the original 17 sources in the cone.

After filtering, most of the 129 ZTF sources will have no counterpart in `gaia_bright`.
The `how` parameter decides what the result looks like for those sources.

In [4]:
gaia_bright = gaia3_sm.query("phot_g_mean_mag < 20")
n_bright = len(gaia_bright.compute())
print(f"Bright Gaia sources (phot_g_mean_mag < 20) : {n_bright}")

Bright Gaia sources (phot_g_mean_mag < 20) : 6


## 4. Inner crossmatch (`how='inner'`, the default)

`how='inner'` keeps **only the rows where a match was found** in both catalogs.
ZTF sources that fall outside the 1-arcsecond search radius of every bright Gaia source
are **silently dropped** from the result.

This is the default, so `ztf22_sm.crossmatch(gaia_bright, radius_arcsec=1.0)` and
`ztf22_sm.crossmatch(gaia_bright, radius_arcsec=1.0, how='inner')` are equivalent.

In [5]:
inner_result = ztf22_sm.crossmatch(
    gaia_bright,
    radius_arcsec=1.0,
    how="inner",
    suffix_method="overlapping_columns",
    log_changes=False,
).compute()

inner_result[["objectid", "nepochs", "source_id", "phot_g_mean_mag", "_dist_arcsec"]]

,objectid,nepochs,source_id,phot_g_mean_mag,_dist_arcsec
_healpix_29,,,,,
2136230511175494507,435313300083253,174,4272461018841219072,15.848508,0.043069
2136230511186576907,435113300001005,377,4272461018841219072,15.848508,0.093870
2136230511186589223,1481108400021268,14,4272461018841219072,15.848508,0.105821
2136230511186591124,1481208400054176,31,4272461018841219072,15.848508,0.111156
2136230511187068011,435213300027173,921,4272461018841219072,15.848508,0.116384
...,...,...,...,...,...


In [6]:
print(f"ZTF sources (left)         : {n_ztf}")
print(f"Bright Gaia sources (right):   {n_bright}")
print(f"Inner result rows          :  {len(inner_result)}  \u2190 {n_ztf - len(inner_result)} ZTF sources silently dropped")

ZTF sources (left)         : 129
Bright Gaia sources (right):   6
Inner result rows          :  23  ← 106 ZTF sources silently dropped


## 5. Left crossmatch (`how='left'`)

`how='left'` keeps **every row from the left catalog**, regardless of whether a match was found.
For ZTF sources that have no bright Gaia counterpart within 1 arcsecond, all Gaia columns in
that row are filled with `<NA>`.

The row count of the result equals the row count of the left catalog (when `n_neighbors=1`).

In [7]:
left_result = ztf22_sm.crossmatch(
    gaia_bright,
    radius_arcsec=1.0,
    how="left",
    suffix_method="overlapping_columns",
    log_changes=False,
).compute()

left_result[["objectid", "nepochs", "source_id", "phot_g_mean_mag", "_dist_arcsec"]]

,objectid,nepochs,source_id,phot_g_mean_mag,_dist_arcsec
_healpix_29,,,,,
2136230497219052831,435313300008704,18,<NA>,<NA>,<NA>
2136230497608086572,435213300074278,15,<NA>,<NA>,<NA>
2136230497630747626,435313300083450,81,<NA>,<NA>,<NA>
...,...,...,...,...,...
2136230511175494507,435313300083253,174,4272461018841219072,15.848508,0.043069
2136230511186576907,435113300001005,377,4272461018841219072,15.848508,0.093870
...,...,...,...,...,...


In [8]:
print(f"ZTF sources (left)         : {n_ztf}")
print(f"Bright Gaia sources (right):   {n_bright}")
print(f"Left result rows           : {len(left_result)}  \u2190 all ZTF sources preserved")

ZTF sources (left)         : 129
Bright Gaia sources (right):   6
Left result rows           : 129  ← all ZTF sources preserved


### 5.1 Identifying unmatched sources

Because unmatched rows have `<NA>` in all right-catalog columns, you can filter them
with a null check on any right-catalog column — for example `source_id`:

In [9]:
n_matched   = left_result["source_id"].notna().sum()
n_unmatched = left_result["source_id"].isna().sum()
print(f"ZTF sources with a bright Gaia match    : {n_matched:3d} ({n_matched/len(left_result):.1%})")
print(f"ZTF sources with no bright Gaia match   : {n_unmatched:3d} ({n_unmatched/len(left_result):.1%})")

ZTF sources with a bright Gaia match    :  23 (17.8%)
ZTF sources with no bright Gaia match   : 106 (82.2%)


## 6. When to choose each

| | `how='inner'` (default) | `how='left'` |
|---|---|---|
| **Result rows** | One per matched pair | One per left-catalog source |
| **Unmatched left sources** | Dropped silently | Kept, right columns are `<NA>` |
| **Typical use** | You only care about confirmed matches | You need to account for every left-catalog source |

**Use `how='inner'`** when you only need sources that appear in both catalogs — for example,
comparing photometry for objects observed by two telescopes, where sources not seen by both are irrelevant.

**Use `how='left'`** when you need to preserve every source from your primary (left) catalog.
Common scenarios include:

- **Completeness analysis**: computing what fraction of your primary sample has a counterpart at another wavelength.
- **Upper-limit studies**: you need a row per source even when no match exists, so you can fill in flux upper limits.
- **Flagging**: you want to add a "matched" boolean column and keep the full primary catalog.

```python
# Add a boolean column flagging which ZTF sources have a bright Gaia counterpart
result = ztf22_sm.crossmatch(gaia_bright, radius_arcsec=1.0, how="left")
result_df = result.compute()
result_df["has_bright_gaia"] = result_df["source_id"].notna()
```

> **Default is `'inner'`**: If the result of `.crossmatch()` has fewer rows than your left catalog,
> the missing rows are unmatched sources that were silently dropped.
> Switch to `how='left'` to see them.

## About

**Last updated on**: May 2026

If you use `lsdb` for published research, please cite following [instructions](https://docs.lsdb.io/en/stable/citation.html).